In [2]:
import sys
from pathlib import Path

# Find project root dynamically
def get_project_root() -> Path:
    try:
        path = Path(__file__).resolve()
        for parent in [path] + list(path.parents):
            if (parent / "requirements.txt").exists() or (parent / "project").exists():
                return parent
    except NameError:
        pass
    path = Path.cwd().resolve()
    for parent in [path] + list(path.parents):
        if (parent / "requirements.txt").exists() or (parent / "project").exists():
            return parent
    return path

ROOT = get_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
import torch.nn as nn
from torch.nn import functional as F
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# parameters
batch_size = 64
block_size = 64
max_iters = 5000
eval_interval = 300
learning_rate = 3e-4

n_embd = 128
n_head = 4
n_layer = 4
dropout = 0.2


# read text file
with open(ROOT / 'TASK 1' / 'input.txt', 'r', encoding='utf-8') as f:
    text = f.read()


# vocabulary
chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}


def encode(s):
    return [stoi[c] for c in s]


def decode(l):
    return ''.join([itos[i] for i in l])


# dataset
data = torch.tensor(encode(text), dtype=torch.long)

n = int(0.9 * len(data))

train_data = data[:n]
test_data = data[n:]


# batching
def get_batch(split):

    data = train_data if split == 'train' else test_data

    ix = torch.randint(
        len(data) - block_size,
        (batch_size,)
    )

    x = torch.stack([
        data[i:i + block_size]
        for i in ix
    ])

    y = torch.stack([
        data[i + 1:i + block_size + 1]
        for i in ix
    ])

    x = x.to(device)
    y = y.to(device)

    return x, y


# SINGLE HEAD
class Head(nn.Module):

    def __init__(self, head_size):

        super().__init__()

        self.key = nn.Linear(
            n_embd,
            head_size,
            bias=False
        )

        self.query = nn.Linear(
            n_embd,
            head_size,
            bias=False
        )

        self.value = nn.Linear(
            n_embd,
            head_size,
            bias=False
        )

        self.register_buffer(
            'tril',
            torch.tril(
                torch.ones(block_size, block_size)
            )
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        B, T, C = x.shape

        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        wei = q @ k.transpose(-2, -1)

        wei = wei * (k.size(-1) ** -0.5)

        wei = wei.masked_fill(
            self.tril[:T, :T] == 0,
            float('-inf')
        )

        wei = F.softmax(wei, dim=-1)

        wei = self.dropout(wei)

        out = wei @ v

        return out


# MULTI HEAD ATTENTION
class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size):

        super().__init__()

        self.heads = nn.ModuleList([
            Head(head_size)
            for _ in range(num_heads)
        ])

        self.proj = nn.Linear(
            num_heads * head_size,
            n_embd
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        out = torch.cat(
            [h(x) for h in self.heads],
            dim=-1
        )

        out = self.proj(out)

        out = self.dropout(out)

        return out


# FEED FORWARD
class FeedForward(nn.Module):

    def __init__(self, n_embd):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(
                n_embd,
                4 * n_embd
            ),

            nn.ReLU(),

            nn.Linear(
                4 * n_embd,
                n_embd
            ),

            nn.Dropout(dropout)
        )

    def forward(self, x):

        return self.net(x)


# TRANSFORMER BLOCK
class Block(nn.Module):

    def __init__(
        self,
        n_embd,
        n_head,
        use_residual=True,
        use_layernorm=True
    ):

        super().__init__()

        self.use_residual = use_residual
        self.use_layernorm = use_layernorm

        head_size = n_embd // n_head

        self.sa = MultiHeadAttention(
            n_head,
            head_size
        )

        self.ffwd = FeedForward(n_embd)

        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):

        # attention input
        x1 = self.ln1(x) if self.use_layernorm else x

        # attention block
        if self.use_residual:
            x = x + self.sa(x1)
        else:
            x = self.sa(x1)

        # feedforward input
        x2 = self.ln2(x) if self.use_layernorm else x

        # feedforward block
        if self.use_residual:
            x = x + self.ffwd(x2)
        else:
            x = self.ffwd(x2)

        return x


# TRANSFORMER MODEL
class TransformerLanguageModel(nn.Module):

    def __init__(
        self,
        use_residual=True,
        use_layernorm=True
    ):

        super().__init__()

        self.token_embedding_table = nn.Embedding(
            vocab_size,
            n_embd
        )

        self.position_embedding_table = nn.Embedding(
            block_size,
            n_embd
        )

        self.blocks = nn.Sequential(

            Block(
                n_embd,
                n_head,
                use_residual,
                use_layernorm
            ),

            Block(
                n_embd,
                n_head,
                use_residual,
                use_layernorm
            ),

            Block(
                n_embd,
                n_head,
                use_residual,
                use_layernorm
            ),

            Block(
                n_embd,
                n_head,
                use_residual,
                use_layernorm
            ),

            nn.LayerNorm(n_embd)
        )

        self.lm_head = nn.Linear(
            n_embd,
            vocab_size
        )

    def forward(self, idx, targets=None):

        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)

        pos_emb = self.position_embedding_table(
            torch.arange(T, device=device)
        )

        x = tok_emb + pos_emb

        x = self.blocks(x)

        logits = self.lm_head(x)

        if targets is None:

            loss = None

        else:

            B, T, C = logits.shape

            logits = logits.view(B * T, C)

            targets = targets.view(B * T)

            loss = F.cross_entropy(
                logits,
                targets
            )

        return logits, loss


# TRAINING FUNCTION
def train_model(
    variant_name,
    use_residual=True,
    use_layernorm=True
):

    print("\n")
    print(f"TRAINING {variant_name}")


    model = TransformerLanguageModel(
        use_residual,
        use_layernorm
    ).to(device)


    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate
    )

    losses = []

    for iter in range(max_iters):

        xb, yb = get_batch('train')

        logits, loss = model(xb, yb)

        optimizer.zero_grad(set_to_none=True)

        loss.backward()

        optimizer.step()

        if iter % eval_interval == 0:

            print(
                f"{variant_name} | "
                f"step {iter}/{max_iters} | "
                f"loss = {loss.item():.4f}"
            )

            losses.append(loss.item())

    return losses


# RUN VARIANT 1
# BASELINE

baseline_losses = train_model(
    variant_name="Baseline",
    use_residual=True,
    use_layernorm=True
)


# RUN VARIANT 2
# NO RESIDUALS

no_residual_losses = train_model(
    variant_name="No Residuals",
    use_residual=False,
    use_layernorm=True
)


# RUN VARIANT 3
# NO LAYERNORM

no_ln_losses = train_model(
    variant_name="No LayerNorm",
    use_residual=True,
    use_layernorm=False
)



# PLOT RESULTS
steps = list(range(
    0,
    max_iters,
    eval_interval
))

plt.figure(figsize=(10, 6))

plt.plot(
    steps,
    baseline_losses,
    label='Baseline'
)

plt.plot(
    steps,
    no_residual_losses,
    label='No Residuals'
)

plt.plot(
    steps,
    no_ln_losses,
    label='No LayerNorm'
)

plt.xlabel("Training Steps")

plt.ylabel("Training Loss")

plt.title("Ablation Experiments")

plt.legend()

plt.grid(True)

plt.show()


FileNotFoundError: [Errno 2] No such file or directory: 'input.txt'

### Variant 1 : Baseline

The baseline transformer trained stably and the loss decreased smoothly throughout training. It achieved the best final performance among all three variants. This happened because both residual connections and LayerNorm were present, which helped maintain stable activations and smooth gradient flow across all transformer blocks.

---

### Variant 2 : No Residual Connections

Removing residual connections caused training to become much worse. The loss decreased slightly at the beginning but quickly stopped improving and stayed much higher than the baseline. Without residual connections, gradients could not flow easily through deeper layers, making optimization difficult. The model struggled to save information across blocks, which affected learning.

---

### Variant 3 : No LayerNorm

Removing LayerNorm made training slightly less stable, but the model still learned reasonably well and achieved a loss close to the baseline. LayerNorm normally helps stabilize activations and gradients during training. However, because residual connections were still present, gradient flow remained strong enough for the network to continue learning effectively.